# Alexandria Audiobook Generator — Google Colab

Run [Alexandria Audiobook2](https://github.com/on22s/alexandria-audiobook2) on a free Google Colab GPU. No local installation required.

> **Note:** This Colab notebook is provided for convenience so you can try Alexandria without local installation. It is not the primary focus of the project — for the best experience, install Alexandria locally via [Pinokio](https://pinokio.computer).

**What this notebook does:**
1. Checks your GPU runtime
2. Installs Alexandria and all dependencies
3. (Optional) Mounts Google Drive to cache TTS models across sessions
4. Starts the Alexandria server and creates a public URL via ngrok

**Before you start:**
- Make sure you've selected a **GPU runtime**: Runtime → Change runtime type → T4 GPU
- You need a free [ngrok account](https://dashboard.ngrok.com/signup) for the tunnel
- You need an LLM server for script generation (see Cell 5 for options)

**Colab T4 GPU (15 GB VRAM) recommended settings:**
- Parallel Workers: 5-10
- Max Chars/Batch: 1500-2000
- Compile Codec: enabled (recommended for speed)

---

## 1. Check GPU Runtime

Verify that a GPU is available. If this cell fails, go to **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected! Go to Runtime → Change runtime type → T4 GPU, then re-run this cell."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print()
if vram_gb < 10:
    print("WARNING: Less than 10 GB VRAM. Batch sizes will be very limited.")
    print("Use Parallel Workers: 2-3 and Max Chars/Batch: 1000 in Setup.")
elif vram_gb < 16:
    print("T4 detected (15 GB). Recommended: Parallel Workers 5-10, Max Chars/Batch 1500-2000.")
else:
    print(f"{vram_gb:.0f} GB VRAM available. You can use higher batch sizes.")

## 2. Install Alexandria

Clones the repository and installs all Python dependencies. This takes 2-3 minutes.

In [ ]:
import os
import subprocess
import sys

ALEXANDRIA_DIR = "/content/Alexandria"

# Stop on clone/update errors before attempting installation.
if not os.path.exists(ALEXANDRIA_DIR):
    subprocess.run(["git", "clone", "https://github.com/on22s/alexandria-audiobook2.git",
                    ALEXANDRIA_DIR], check=True)
else:
    print(f"Alexandria already cloned at {ALEXANDRIA_DIR}")
    subprocess.run(["git", "-C", ALEXANDRIA_DIR, "pull", "--ff-only"], check=True)

# Resolve application and TTS requirements together in this kernel's Python.
# Keep pip's diagnostics visible and stop immediately if installation fails.
subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                os.path.join(ALEXANDRIA_DIR, "app", "requirements.txt"),
                "qwen-tts==0.1.1", "pyngrok"], check=True)

print()
print("Installation complete.")


## 3. Mount Google Drive (Optional)

TTS models are **~3.5 GB each** and download on first use. By default, they're lost when the Colab session ends.

Mounting Google Drive caches models persistently so you don't re-download them every session.

**Skip this cell** if you don't want to use Drive storage (models will re-download each session).

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

# Set HuggingFace cache to Google Drive
hf_cache = "/content/drive/MyDrive/.cache/huggingface"
os.makedirs(hf_cache, exist_ok=True)
os.environ["HF_HOME"] = hf_cache

print(f"HuggingFace cache set to: {hf_cache}")
print("Models will persist across Colab sessions.")

## 4. Configure ngrok Tunnel

ngrok creates a public URL so you can access the Alexandria web UI from your browser.

1. Sign up for a free account at [ngrok.com](https://dashboard.ngrok.com/signup)
2. Copy your auth token from [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)
3. Paste it below and run the cell

In [ ]:
NGROK_AUTH_TOKEN = ""  # @param {type:"string"}

if not NGROK_AUTH_TOKEN:
    raise ValueError(
        "Please set your ngrok auth token above.\n"
        "Get one free at: https://dashboard.ngrok.com/get-started/your-authtoken"
    )

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("ngrok configured.")

## 5. Start Alexandria

This cell:
1. Writes a default config (local TTS mode, auto GPU device)
2. Starts the Alexandria server in the background
3. Opens an ngrok tunnel to the web UI

**After running this cell**, click the ngrok URL to open the Alexandria web UI.

### LLM Setup

Alexandria needs an LLM server for script generation. In the web UI **Setup tab**, configure one of:

| Provider | Base URL | API Key |
|----------|----------|--------|
| OpenAI | `https://api.openai.com/v1` | Your OpenAI API key |
| DeepSeek | `https://api.deepseek.com/v1` | Your DeepSeek API key |
| OpenRouter | `https://openrouter.ai/api/v1` | Your OpenRouter API key |
| Ollama (see Cell 6) | `http://localhost:11434/v1` | `local` |

Or any other OpenAI-compatible API.

In [ ]:
import json
import os
import subprocess
import time
import requests
from pyngrok import ngrok

ALEXANDRIA_DIR = "/content/Alexandria"
APP_DIR = os.path.join(ALEXANDRIA_DIR, "app")
CONFIG_PATH = os.path.join(APP_DIR, "config.json")

# Write default config if none exists
if not os.path.exists(CONFIG_PATH):
    config = {
        "llm": {
            "base_url": "http://localhost:11434/v1",
            "api_key": "local",
            "model_name": ""
        },
        "tts": {
            "mode": "local",
            "device": "auto",
            "language": "English",
            "parallel_workers": 8,
            "compile_codec": True,
            "sub_batch_enabled": True,
            "sub_batch_min_size": 4,
            "sub_batch_ratio": 5,
            "sub_batch_max_chars": 2000
        }
    }
    with open(CONFIG_PATH, "w") as f:
        json.dump(config, f, indent=2)
    print("Default config written (local TTS, auto GPU).")
else:
    print("Existing config.json found, keeping it.")

# Always drain the pipe, even when the optional viewer is stopped.
import signal
import threading
import sys

class _ServerLogs:
    MAX_BYTES = 1024 * 1024  # Current file + two backups: at most 3 MiB.

    def __init__(self, path):
        self.path = path
        self.lock = threading.Lock()
        self.error = None
        self.thread = None
        # Reuse only these notebook-owned files; keep the previous run's tail.
        sizes = [os.path.getsize(p) for p in self.paths() if os.path.exists(p)]
        if any(size > self.MAX_BYTES for size in sizes):
            raise RuntimeError("Existing Colab log exceeds the cap; move it aside before starting.")
        self.total = sum(sizes)
        self.file = open(path, "ab", buffering=0)

    def paths(self):
        return [self.path + ".2", self.path + ".1", self.path]

    def start(self, process):
        self.thread = threading.Thread(target=self.drain, args=(process,), daemon=True)
        self.thread.start()

    def drain(self, process):
        try:
            while True:
                data = process.stdout.read(8192)
                if not data:
                    break
                with self.lock:
                    if self.error:
                        continue  # A disk error must not leave the child pipe blocked.
                    try:
                        while data:
                            size = self.file.tell()
                            if size == self.MAX_BYTES:
                                self.file.close()  # Reopen after rename: no stale inode writes.
                                for src, dst in [(self.path + ".1", self.path + ".2"),
                                                 (self.path, self.path + ".1")]:
                                    if os.path.exists(src):
                                        os.replace(src, dst)
                                self.file = open(self.path, "wb", buffering=0)
                                size = 0
                            part, data = data[:self.MAX_BYTES - size], data[self.MAX_BYTES - size:]
                            written = self.file.write(part)
                            if written != len(part):
                                raise OSError("Incomplete server log write")
                            self.total += written
                    except OSError as exc:
                        # Reported by finish() and the viewer; a background thread
                        # printing here would land in whichever cell is running.
                        self.error = f"logging failed, later output discarded: {exc}"
        except Exception as exc:
            self.error = f"log drain failed: {exc}"
            if process.poll() is None:
                process.terminate()  # Do not silently leave a running child with no reader.
        finally:
            process.stdout.close()
            self.file.close()

    def read(self, cursor, limit=65536):
        with self.lock:
            paths = [p for p in self.paths() if os.path.exists(p)]
            sizes = [os.path.getsize(p) for p in paths]
            first = self.total - sum(sizes)
            skipped = max(0, first - cursor)
            cursor = max(cursor, first)
            chunks = []
            for path, size in zip(paths, sizes):
                offset = max(0, cursor - first)
                if offset < size and limit:
                    with open(path, "rb") as f:
                        f.seek(offset)
                        chunk = f.read(min(limit, size - offset))
                    chunks.append(chunk)
                    cursor += len(chunk)
                    limit -= len(chunk)
                first += size
            return cursor, b"".join(chunks), skipped

    def finish(self):
        if self.thread is None or self.thread.ident is None:
            self.file.close()
        else:
            self.thread.join(timeout=5)
            if self.thread.is_alive():
                raise RuntimeError(
                    "Server log reader is still active; refusing to reuse its log files. "
                    "Stop the stray process or reset the runtime before restarting."
                )
        if self.error:
            print(f"WARNING: Server log error: {self.error}")


def _signal_server_group(process, sig):
    # The server runs in its own session (start_new_session), so its pid is the
    # group id. Signalling the group also stops workers it started, which share
    # its stdout pipe and would otherwise keep the log reader from finishing.
    try:
        os.killpg(process.pid, sig)
    except ProcessLookupError:
        pass


def _stop_server():
    process = globals().get("server_process")
    if process is not None:
        _signal_server_group(process, signal.SIGTERM)
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            pass
        _signal_server_group(process, signal.SIGKILL)  # stragglers in the group
        process.wait(timeout=5)
    logs = globals().get("server_logs")
    if logs is not None:
        # A failed reader start still leaves a parent-side pipe to close.
        if process is not None and (logs.thread is None or logs.thread.ident is None):
            process.stdout.close()
        # Also check a leftover reader when an earlier cell lost the handle.
        # Do not allow a new writer on these paths until the old one has exited.
        logs.finish()
    if process is not None:
        print(f"Server stopped (exit code {process.returncode}).")
    else:
        print("Server was not started.")

# Stop the previous run before replacing its handle or log reader.
_stop_server()
print("Starting Alexandria server...")
LOG_PATH = os.path.join(ALEXANDRIA_DIR, "colab-server.log")
server_logs = _ServerLogs(LOG_PATH)
server_process = None
try:
    server_process = subprocess.Popen(
        [sys.executable, "-u", "app.py"], cwd=APP_DIR,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0,
        start_new_session=True
    )
    server_logs.start(server_process)
except BaseException:
    # Popen may have succeeded before reader startup failed or was interrupted.
    # Keep the handle if cleanup fails, so Stop or a later Start can retry.
    _stop_server()
    server_process = None
    raise
print(f"Server logs: {LOG_PATH} (at most 3 MiB including two backups).")

# Do not advertise a tunnel when this process exits or startup times out.
try:
    for i in range(30):
        if server_process.poll() is not None:
            raise RuntimeError(f"Server exited with code {server_process.returncode}")
        try:
            r = requests.get("http://127.0.0.1:4200/api/config", timeout=2)
            if r.status_code == 200:
                print("Server is running.")
                break
        except requests.RequestException:
            pass
        time.sleep(2)
    else:
        raise RuntimeError("Server did not become ready within the startup checks")
except BaseException:
    _stop_server()
    _, output, _ = server_logs.read(max(0, server_logs.total - 65536))
    print(output.decode("utf-8", errors="replace"), end="")
    raise

# Replace the previous tunnel when rerunning Start.
try:
    ngrok.kill()
    public_url = ngrok.connect(4200)
except BaseException:
    _stop_server()
    raise
print()
print("=" * 60)
print(f"Alexandria is running at: {public_url}")
print("=" * 60)
print()
print("Open the URL above in your browser.")
print("Configure your LLM in the Setup tab before generating scripts.")
print()
print("First TTS generation will download the model (~3.5 GB).")
print("Run View Server Logs for download progress.")

## 6. (Optional) Install Ollama for Local LLM

If you don't have a cloud LLM API, you can run Ollama on Colab for script generation.

**Important — VRAM sharing:** Ollama and TTS both use the T4 GPU. A 7B model uses ~5 GB VRAM, leaving only ~10 GB for TTS. This **will cause out-of-memory crashes** during batch generation, especially with LoRA voices.

**Workflow:** Use Ollama for script generation, then **run Cell 6b to stop Ollama** before starting TTS batch generation. The LLM is only needed during script generation.

**Skip this cell** if you're using a cloud API (OpenAI, DeepSeek, etc.) — that's the easiest option on Colab.

In [ ]:
import subprocess
import time

OLLAMA_MODEL = "qwen2.5:7b"  # @param {type:"string"}

# Install zstd (required by Ollama installer)
!apt-get install -y -qq zstd

# Install Ollama
print("Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in background
print("Starting Ollama server...")
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)

# Pull the model
print(f"Pulling {OLLAMA_MODEL} (this may take a few minutes)...")
!ollama pull {OLLAMA_MODEL}

print()
print(f"Ollama is running with {OLLAMA_MODEL}.")
print()
print("In Alexandria Setup tab, configure:")
print(f"  LLM Base URL: http://localhost:11434/v1")
print(f"  API Key: local")
print(f"  Model Name: {OLLAMA_MODEL}")

## 6b. Stop Ollama (Free VRAM for TTS)

**Run this cell after script generation is complete**, before starting batch TTS generation. This frees ~5 GB of VRAM that Ollama was using for the LLM.

You do NOT need to run this if you used a cloud LLM API instead of Ollama.

In [ ]:
import subprocess

# Stop Ollama server to free GPU memory for TTS
try:
    subprocess.run(["pkill", "-f", "ollama"], timeout=5)
    print("Ollama stopped. GPU memory freed for TTS.")
except:
    print("Ollama was not running.")

# Verify VRAM is freed
import torch
if torch.cuda.is_available():
    free_mem = torch.cuda.mem_get_info()[0] / 1e9
    total_mem = torch.cuda.mem_get_info()[1] / 1e9
    print(f"VRAM: {free_mem:.1f} GB free / {total_mem:.1f} GB total")

## 7. View Server Logs

Run this cell to see real-time server output (model loading, generation progress, errors).

**Interrupt the cell** (stop button) to stop viewing logs. The server keeps running.

Logs are saved to `/content/Alexandria/colab-server.log` with two rotated backups (3 MiB maximum). Restarting preserves the recent tail. Viewing resumes from the last 64 KiB; older output is discarded as files rotate.

In [ ]:
# Show the last 64 KiB, then follow; interrupting only stops this viewer.
import codecs

if "server_logs" not in globals():
    print("Start the server first.")
else:
    logs = server_logs
    cursor = max(0, logs.total - 65536)
    decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
    try:
        while True:
            cursor, data, skipped = logs.read(cursor)
            if skipped:
                print(f"\n[Log rotation discarded {skipped} older bytes]")
                decoder.reset()
            if data:
                print(decoder.decode(data), end="", flush=True)
            elif logs.thread is None or not logs.thread.is_alive():
                print(decoder.decode(b"", final=True), end="")
                if logs.error:
                    print(f"\nServer log error: {logs.error}")
                print(f"\nServer exited (code {server_process.poll()}).")
                break
            else:
                time.sleep(0.25)
    except KeyboardInterrupt:
        print("\nStopped viewing logs. Server logging continues in the background.")


## 8. Stop Server

Run this cell when you're done to clean up.

In [ ]:
from pyngrok import ngrok

# Reap the server even if ngrok cleanup fails.
try:
    ngrok.kill()
    print("ngrok tunnel closed.")
finally:
    if "_stop_server" in globals():
        _stop_server()
    else:
        print("Server was not started.")

# Kill Ollama if running
try:
    ollama_process.terminate()
    print("Ollama stopped.")
except:
    pass